In [1]:
import os
os.environ["LD_LIBRARY_PATH"] = "/home/pcarragh/miniconda3/envs/lmms-finetune/lib/python3.10/site-packages/nvidia/nvjitlink/lib:$LD_LIBRARY_PATH"

from eval_1022 import *
import json
import numpy as np

exp_name = "llava"
path = "/home/pcarragh/dev/webqa/LLaVA/WebQA_train_val_color_gpt_matched.json"
eval_data = json.load(open(path, "r"))

print(len(eval_data))
sum([len(v['A_perturbed']) for k, v in eval_data.items()])

572


859

In [2]:
from transformers import AutoProcessor, AutoTokenizer, LlavaForConditionalGeneration
import torch
from PIL import Image
from importlib import reload
import convert_webqa_data
reload(convert_webqa_data)
from convert_webqa_data import get_prompt
from PIL import Image
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration

original_model_id = "llava-hf/llava-1.5-7b-hf"
# original_model_id = "llava-hf/llava-interleave-qwen-7b-hf"
# model_id = "/home/pcarragh/dev/webqa/lmms-finetune/checkpoints/llava-1.5-7b_lora-True_qlora-False/" # original webqa, no perturbations
model_id = "/home/pcarragh/dev/webqa/lmms-finetune/checkpoints/llava-1.5-7b_v2_lora-True_qlora-False/" # perturbed webqa <RET>

model_path = model_id
model = LlavaForConditionalGeneration.from_pretrained(
    original_model_id, # model_id, 
    torch_dtype=torch.float16, 
    # low_cpu_mem_usage=True, 
).to(0)
processor = AutoProcessor.from_pretrained(original_model_id)


def fetch_webqa_sample(key, image_paths, reverse_images = False):
    images = []
    if reverse_images:
        image_paths = image_paths[::-1]
        
    for image_path in image_paths:
        try:
            image_path_int = int(image_path)
        except:
            image_path_int = None
        
        if image_path_int:
            images.append(load_webqa_image(image_path))
        else:
            images.append(Image.open(image_path).convert("RGB"))

    query = get_prompt(eval_data, key, reverse_images)
    return images, query

def llava_eval_on_webqa_sample(key, image_paths, reverse_images = False):
    images, query = fetch_webqa_sample(key, image_paths, reverse_images)
    query = "HUMAN:" + query + ' \nGPT:'
    # print(query) 
    inputs = processor(images=images, text=query, return_tensors='pt').to(0, torch.float16)
    output = model.generate(
        **inputs, 
        max_new_tokens=50,
        do_sample=False, 
        # max_length=100,
        num_return_sequences=1,
        temperature=0.0,
    )
    return processor.decode(output[0][2:], skip_special_tokens=True)

def webqa_accuracy(answer, label, Qcate):
    if Qcate == 'color':
        F1_avg, F1_max, EM, RE_avg, PR_avg = compute_vqa_metrics([answer], label[0], "", color_set)
    elif Qcate == 'shape': 
        F1_avg, F1_max, EM, RE_avg, PR_avg = compute_vqa_metrics([answer], label[0], "", shape_set)
    elif Qcate == 'yesno': 
        F1_avg, F1_max, EM, RE_avg, PR_avg = compute_vqa_metrics([answer], label[0], "", yesno_set)
    elif Qcate == 'number': 
        F1_avg, F1_max, EM, RE_avg, PR_avg = compute_vqa_metrics([answer], label[0], "", {"NUMBER"})
    else:
        return None
    return (F1_avg, F1_max, EM, RE_avg, PR_avg)

def accuracy_agg_results(qa_results):
    single_image_keys = [k for k in qa_results.keys() if len(eval_data[k]['img_posFacts']) == 1]
    two_image_keys = [k for k in qa_results.keys() if len(eval_data[k]['img_posFacts']) == 2]

    single_acc = np.mean([PR_avg for key, (F1_avg, F1_max, EM, RE_avg, PR_avg) in qa_results.items() if key in single_image_keys])
    two_image_acc = np.mean([PR_avg for key, (F1_avg, F1_max, EM, RE_avg, PR_avg) in qa_results.items() if key in two_image_keys])
    avr_acc = np.mean([PR_avg for key, (F1_avg, F1_max, EM, RE_avg, PR_avg) in qa_results.items()])
    return (single_acc, two_image_acc, avr_acc)

def accuracy_agg_generated_results(qa_results):
    single_image_keys = [k for k in qa_results.keys() if len(eval_data[k]['img_posFacts']) == 1]
    two_image_keys = [k for k in qa_results.keys() if len(eval_data[k]['img_posFacts']) == 2]

    single_acc = np.mean([PR_avg for key, dict in qa_results.items() if key in single_image_keys for idx, (_,_,_,_,PR_avg) in dict.items()])
    two_image_acc = np.mean([PR_avg for key, dict in qa_results.items() if key in two_image_keys for idx, (_,_,_,_,PR_avg) in dict.items()])
    avr_acc = np.mean([PR_avg for key, dict in qa_results.items() for idx, (_,_,_,_,PR_avg) in dict.items()])
    
    return (single_acc, two_image_acc, avr_acc)


/home/pcarragh/miniconda3/envs/lmms-finetune/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: /home/pcarragh/miniconda3/envs/lmms-finetune/lib/python3.10/site-packages/torch/lib/../../nvidia/cusparse/lib/libcusparse.so.12: undefined symbol: __nvJitLinkComplete_12_4, version libnvJitLink.so.12

In [5]:
test_key = list(eval_data.keys())[0]
test_image = str(eval_data[test_key]['img_posFacts'][0]['image_id'])
ans = llava_eval_on_webqa_sample(test_key, [test_image], True)
ans.split('GPT:')[1].strip()

Expanding inputs for image tokens in LLaVa should be done in processing. Please add `patch_size` and `vision_feature_select_strategy` to the model's processing config or set directly with `processor.patch_size = {{patch_size}}` and processor.vision_feature_select_strategy = {{vision_feature_select_strategy}}`. Using processors without these attributes in the config is deprecated and will throw an error in v4.47.
/home/pcarragh/miniconda3/envs/lmms-finetune/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Expanding inputs for image tokens in LLaVa should be done in processing. Please add `patch_size` and `vision_feature_select_strategy` to the model's processing config or s

'The facade of bakery Sattin et Fils in Rethel, France is red.'

### LLAVA baseline on original images that have perturbations

In [49]:
from tqdm import tqdm

llava_results_baseline = {}
for k in tqdm(list(eval_data.keys())):
    # question = eval_data[k]['Q']
    image_files = [str(img_data['image_id']) for img_data in eval_data[k]['img_posFacts']]
    try:
        answer = llava_eval_on_webqa_sample(k, image_files)
    except :
        answer = 'ERROR: llava finetune failed'
    label = eval_data[k]['A']
    eval_data[k]['A_llava'] = answer
    Qcate = eval_data[k]['Qcate'].lower()
    llava_results_baseline[k] = webqa_accuracy(answer, label, Qcate)

print(accuracy_agg_results(llava_results_baseline))

100%|██████████| 572/572 [33:22<00:00,  3.50s/it]

(np.float64(0.7188961988304093), np.float64(0.8258620689655172), np.float64(0.7405885780885781))


### Correct despite blank image

In [6]:
llava_results_blank = {}
for k in tqdm(list(eval_data.keys())):
    # question = eval_data[k]['Q']
    image_files = ['/home/pcarragh/dev/webqa/LLaVA/playground/counterfactual_exp/BLANK.jpg'] * len(eval_data[k]['img_posFacts'])

    try:
        answer = llava_eval_on_webqa_sample(k, image_files)
    except:
        answer = 'ERROR: llava failed'
    label = eval_data[k]['A']
    eval_data[k]['A_llava_blank'] = answer
    Qcate = eval_data[k]['Qcate'].lower()
    llava_results_blank[k] = webqa_accuracy(answer, label, Qcate)
print(accuracy_agg_results(llava_results_blank))

100%|██████████| 572/572 [02:03<00:00,  4.62it/s]

(np.float64(0.0668859649122807), np.float64(0.02586206896551724), np.float64(0.05856643356643357))


: 

### Correct despite perturbation

In [63]:
llava_results_perturbed_original_label = {}
llava_results_perturbed_generated_label = {}

for k in tqdm(list(eval_data.keys())):
    llava_results_perturbed_original_label[k] = {}
    llava_results_perturbed_generated_label[k] = {}
    eval_data[k]['A_perturbed_llava'] = {}
    # question = eval_data[k]['Q']
    for idx, label in eval_data[k]['A_perturbed'].items():
        image_id = eval_data[k]['img_posFacts'][0]['image_id']
        # TODO: add image 2 (image loader can't handle this now)
        # image_files = ','.join([str(img_data['image_id']) for img_data in eval_data[k]['img_posFacts']])
        perturbed_path = f"/home/pcarragh/dev/webqa/segment/Inpaint-Anything/results/webqa/{eval_data[k]['split']}/{str(image_id)}_{k}_{idx}.jpeg"
        image_files = [perturbed_path]
        if len(eval_data[k]['img_posFacts']) > 1:
            image_files.append(eval_data[k]['img_posFacts'][1]['image_id'])
        
        try:
            answer = llava_eval_on_webqa_sample(k, image_files)
        except:
            answer = 'ERROR: llava failed'

        original_label = eval_data[k]['A']
        eval_data[k]['A_perturbed_llava'][idx] = answer
        Qcate = eval_data[k]['Qcate'].lower()
        llava_results_perturbed_original_label[k][idx] = webqa_accuracy(answer, original_label, Qcate)
        llava_results_perturbed_generated_label[k][idx] = webqa_accuracy(answer, [label], Qcate)
        
print(accuracy_agg_generated_results(llava_results_perturbed_original_label))
print(accuracy_agg_generated_results(llava_results_perturbed_generated_label))

100%|██████████| 572/572 [51:21<00:00,  5.39s/it]  


In [4]:
llava_results_perturbed_original_label_reversed = {}
llava_results_perturbed_generated_label_reversed = {}

for k in tqdm(list(eval_data.keys())):
    llava_results_perturbed_original_label_reversed[k] = {}
    llava_results_perturbed_generated_label_reversed[k] = {}
    eval_data[k]['A_perturbed_llava'] = {}
    # question = eval_data[k]['Q']
    for idx, label in eval_data[k]['A_perturbed'].items():
        image_id = eval_data[k]['img_posFacts'][0]['image_id']
        # TODO: add image 2 (image loader can't handle this now)
        # image_files = ','.join([str(img_data['image_id']) for img_data in eval_data[k]['img_posFacts']])
        perturbed_path = f"/home/pcarragh/dev/webqa/segment/Inpaint-Anything/results/webqa/{eval_data[k]['split']}/{str(image_id)}_{k}_{idx}.jpeg"
        image_files = [perturbed_path]
        if len(eval_data[k]['img_posFacts']) > 1:
            image_files.append(eval_data[k]['img_posFacts'][1]['image_id'])
        
        try:
            answer = llava_eval_on_webqa_sample(k, image_files, True)
        except:
            answer = 'ERROR: llava failed'

        original_label = eval_data[k]['A']
        eval_data[k]['A_perturbed_llava'][idx] = answer
        Qcate = eval_data[k]['Qcate'].lower()
        llava_results_perturbed_original_label_reversed[k][idx] = webqa_accuracy(answer, original_label, Qcate)
        llava_results_perturbed_generated_label_reversed[k][idx] = webqa_accuracy(answer, [label], Qcate)
        
print(accuracy_agg_generated_results(llava_results_perturbed_original_label_reversed))
print(accuracy_agg_generated_results(llava_results_perturbed_generated_label_reversed))

  0%|          | 0/572 [00:00<?, ?it/s]Expanding inputs for image tokens in LLaVa should be done in processing. Please add `patch_size` and `vision_feature_select_strategy` to the model's processing config or set directly with `processor.patch_size = {{patch_size}}` and processor.vision_feature_select_strategy = {{vision_feature_select_strategy}}`. Using processors without these attributes in the config is deprecated and will throw an error in v4.47.


/home/pcarragh/miniconda3/envs/lmms-finetune/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Expanding inputs for image tokens in LLaVa should be done in processing. Please add `patch_size` and `vision_feature_select_strategy` to the model's processing config or set directly with `processor.patch_size = {{patch_size}}` and processor.vision_feature_select_strategy = {{vision_feature_select_strategy}}`. Using processors without these attributes in the config is deprecated and will throw an error in v4.47.
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)
100%|██████████| 572/572 [50:21<00:00,  5.28s/it]  

(np.float64(0.15088985088985088), np.float64(0.49377510040160644), np.float64(0.21715172681412495))
(np.float64(0.6501924001924002), np.float64(0.4143574297188755), np.float64(0.6046177726038029))


In [12]:
llava_results_perturbed_original_label_double_gen = {}
llava_results_perturbed_generated_label_double_gen = {}

for k in tqdm(list(eval_data.keys())):
    llava_results_perturbed_original_label_double_gen[k] = {}
    llava_results_perturbed_generated_label_double_gen[k] = {}
    eval_data[k]['A_perturbed_llava_reversed'] = {}
    # question = get_prompt(k, True)
    for idx, label in eval_data[k]['A_perturbed'].items():
        image_id = eval_data[k]['img_posFacts'][0]['image_id']
        image_files = f"/home/pcarragh/dev/webqa/segment/Inpaint-Anything/results/webqa/{eval_data[k]['split']}/{str(image_id)}_{k}_{idx}.jpeg"
        if len(eval_data[k]['img_posFacts']) > 1:
            # image_2 = eval_data[k]['img_posFacts'][1]['image_id']
            image_files = [image_files,image_files]
        else:
            continue
        # try:
        answer = llava_eval_on_webqa_sample(k, image_files)
        # except:
        #     answer = 'ERROR: llava failed'

        original_label = eval_data[k]['A']
        eval_data[k]['A_perturbed_llava_reversed'][idx] = answer
        Qcate = eval_data[k]['Qcate'].lower()
        llava_results_perturbed_original_label_double_gen[k][idx] = webqa_accuracy(answer, original_label, Qcate)
        llava_results_perturbed_generated_label_double_gen[k][idx] = webqa_accuracy(answer, [label], Qcate)

print(accuracy_agg_generated_results(llava_results_perturbed_original_label_double_gen))
print(accuracy_agg_generated_results(llava_results_perturbed_generated_label_double_gen))

100%|██████████| 572/572 [10:05<00:00,  1.06s/it]

(np.float64(nan), np.float64(0.2038152610441767), np.float64(0.2038152610441767))
(np.float64(nan), np.float64(0.6395582329317269), np.float64(0.6395582329317269))


: 

In [65]:
import pandas as pd

baseline_accs = accuracy_agg_results(llava_results_baseline)
blank_accs = accuracy_agg_results(llava_results_blank)
perturbed_original_label_accs = accuracy_agg_generated_results(llava_results_perturbed_original_label)
perturbed_generated_label_acc = accuracy_agg_generated_results(llava_results_perturbed_generated_label)

columns = ['experiment name', 'single_image', 'two_image', 'average']
accuracy_agg_df = pd.DataFrame(columns=columns)
accuracy_agg_df['experiment name'] = ['baseline', 'blank', 'perturbed_original_label', 'perturbed_generated_label']
accuracy_agg_df['single_image'] = [baseline_accs[0], blank_accs[0], perturbed_original_label_accs[0], perturbed_generated_label_acc[0]]
accuracy_agg_df['two_image'] = [baseline_accs[1], blank_accs[1], perturbed_original_label_accs[1], perturbed_generated_label_acc[1]]
accuracy_agg_df['average'] = [baseline_accs[2], blank_accs[2], perturbed_original_label_accs[2], perturbed_generated_label_acc[2]]
accuracy_agg_df.to_csv("{exp_name}.csv", index=False)
accuracy_agg_df


OSError: Cannot save file into a non-existent directory: 'results'

### Correct despite blank image and perturbation

In [50]:
# for questions that llava gets right regardless of having a blank image, check accuracy on counterfactual examples
perturbation_ignored = {key:idx for key, dict in llava_results_perturbed_original_label.items() for idx, (_,_,_,_,acc) in dict.items() if acc > 0}
# and accuray on llava_results_perturbed_generated_label is 0
perturbation_ignored = {key:idx for key, idx in perturbation_ignored.items() if llava_results_perturbed_generated_label[key][idx][4] == 0}
blank_ignored = set([k for k, (_,_,_,_,acc) in llava_results_blank.items() if acc > 0])
parametric_ans_keys = set(perturbation_ignored.keys()).intersection(blank_ignored)
len(parametric_ans_keys)

33

In [51]:
for key in parametric_ans_keys:
    print(f"Key: {key}, Q: {eval_data[key]['Q']}")
    print(f"A: {eval_data[key]['A']}")
    print(f"A_llava: {eval_data[key]['A_llava']}")
    print(f"A_llava_blank: {eval_data[key]['A_llava_blank']}")
    idx = perturbation_ignored[key]
    print(f"A_perturbed_llava: {eval_data[key]['A_perturbed_llava'][idx]}")
    
    print(f"{idx}, A_perturbed: {eval_data[key]['A_perturbed'][idx]}")
    # , A_llava_blank: {eval_data[key]['A_llava_blank']}, A_perturbed_llava: {eval_data[key]['A_perturbed_llava']}")

Key: d5ca49d80dba11ecb1e81171463288e9, Q: "What color are the walls at The Blue Hour Contemporary Art Gallery in Vancouver?"
A: ['"The walls at The Blue Hour Contemporary Art Gallery in Vancouver are white."']
A_llava: The walls at The Blue Hour Contemporary Art Gallery in Vancouver are white.
A_llava_blank: The walls at The Blue Hour Contemporary Art Gallery in Vancouver are white.
A_perturbed_llava: The walls at The Blue Hour Contemporary Art Gallery in Vancouver are black.
2, A_perturbed: gray
Key: d5cf04aa0dba11ecb1e81171463288e9, Q: "What color is the word across the top floor of the Enbridge Centre?"
A: ['"The word across the top floor of the Enbridge Centre is white."']
A_llava: The word across the top floor of the Enbridge Centre is yellow.
A_llava_blank: The word "Enbridge" is white.
A_perturbed_llava: The word across the top floor of the Enbridge Centre is white.
0, A_perturbed: teal
Key: d5c8efca0dba11ecb1e81171463288e9, Q: "What color feathers surround more of the eye of th

: 